# Projeto Fictus | Análise Financeira — Bloco 3: Rentabilidade Financeira

---

## Pergunta Central do Bloco
> **A estrutura financeira da empresa-alvo gera rentabilidade real após o custo do risco — ou o crescimento está mascarando uma erosão silenciosa de margem que o comprador herdaria?**

---

## Contexto do Bloco

O Bloco 1 mapeou a estrutura de recebimento e o spread capturado por intermediários. O Bloco 2 caracterizou o risco: sua concentração, volatilidade e o impacto de choques sobre o resultado financeiro. Este bloco fecha a análise econômica: **o spread disponível compensa o custo do risco que a empresa-alvo carrega — e o que isso significa para o comprador?**

Focar apenas no spread bruto sem descontar o custo do risco, do capital imobilizado e da operação de gestão de crédito leva a conclusões equivocadas sobre a saúde financeira real da operação. Este bloco constrói o spread líquido e o confronta com o custo de oportunidade do capital imobilizado.

**Este bloco investiga:**
1. Quanto da margem atual é capturado por intermediários financeiros?
2. O spread bruto é suficiente para cobrir o custo do risco — após os descontos necessários?
3. O retorno líquido compensa o risco assumido pela operação — e o que isso implica para a decisão de compra?
4. Em quais cenários a rentabilidade se deteriora rapidamente — e quais são os gatilhos críticos?
5. O spread é distribuído uniformemente ao longo do ano ou é concentrado nos meses de maior volume — e isso afeta a estabilidade financeira da operação?

---

## Nota sobre Premissas
Todas as premissas de custo estão declaradas e são modificáveis. Altere os valores e toda a análise se recalcula automaticamente.

---


## Configuração e Carregamento

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import matplotlib.patches as mpatches
import seaborn as sns
from scipy import stats
import warnings
from pathlib import Path

try:
    _base = Path(__file__).resolve().parent
except NameError:
    _base = Path().resolve()
def _find_base(start: Path) -> Path:
    for p in [start, start.parent, start.parent.parent]:
        if (p / "data").exists() or (p / "notebooks").exists():
            return p
    return start
BASE_DIR = _find_base(_base)
DIR_FIN     = BASE_DIR / "data" / "finance"
DIR_EXPORTS = BASE_DIR / "exports"
DIR_EXPORTS.mkdir(parents=True, exist_ok=True)
warnings.filterwarnings("ignore")

COR_RECEITA  = "#1B4F72"
COR_MARGEM   = "#27AE60"
COR_ALERTA   = "#C0392B"
COR_NEUTRO   = "#7F8C8D"
COR_DESTAQUE = "#E67E22"
COR_ROXO     = "#8E44AD"

sns.set_theme(style="whitegrid", font_scale=1.0)
plt.rcParams.update({
    "figure.dpi": 150, "savefig.dpi": 150, "savefig.bbox": "tight",
    "font.family": "sans-serif",
    "axes.spines.top": False, "axes.spines.right": False,
})
def fmt_brl(x, pos=None):
    if abs(x) >= 1_000_000: return f"R$ {x/1_000_000:.1f}M"
    if abs(x) >= 1_000:     return f"R$ {x/1_000:.0f}K"
    return f"R$ {x:.0f}"
def fmt_pct(x, pos=None): return f"{x:.1f}%"
def salvar(fig, nome):
    caminho = DIR_EXPORTS / f"{nome}.png"
    fig.savefig(caminho)
    print(f"  → Salvo: {caminho.name}")

def ler(f, **kw):
    df = pd.read_csv(DIR_FIN / f, low_memory=False, **kw)
    df.columns = df.columns.str.strip()
    return df

fin_fato   = ler("fin_fato.csv")
fin_mensal = ler("fin_mensal.csv")
fin_pag    = ler("fin_pagamento.csv")
fin_faixa  = ler("fin_faixa.csv")

for col in ["preco", "numero_parcelas", "pmr_ajustado", "spread_intermediario",
            "capital_em_aberto", "anomalia_pagamento"]:
    if col in fin_fato.columns:
        fin_fato[col] = pd.to_numeric(fin_fato[col], errors="coerce")

# ─── Premissas de custo de internalização — todas auditáveis ─────────────────
# Custo operacional de gestão de crédito (equipe, sistema, compliance)
# Premissa declarada — modificável para análise de sensibilidade
CUSTO_OPERACIONAL_MENSAL = 25_000     # R$/mês — equipe mínima de crédito
# Custo de capital (taxa de oportunidade do capital imobilizado)
CUSTO_CAPITAL_AM         = 0.0120     # 1,2% a.m. = ~15% a.a. (CDI + spread)
# Taxa esperada de inadimplência (com base no Bloco 2)
TAXA_INADIMPLENCIA       = fin_fato["anomalia_pagamento"].mean()
# Custo de estrutura de operação de crédito (sistemas, regulatório, contratos)
# Premissa declarada — modificável para análise de sensibilidade
CUSTO_SETUP              = 150_000    # R$ — investimento de referência

print("Premissas de custo carregadas:")
print(f"  Custo operacional mensal : R$ {CUSTO_OPERACIONAL_MENSAL:,.0f}")
print(f"  Custo de capital (a.m.)  : {CUSTO_CAPITAL_AM*100:.2f}%")
print(f"  Taxa de inadimplência    : {TAXA_INADIMPLENCIA*100:.2f}% (derivada do Bloco 2)")
print(f"  Custo de setup           : R$ {CUSTO_SETUP:,.0f}")


---

## Análise 1 — Spread Capturado por Intermediários

> *"O ponto de partida para a avaliação da rentabilidade financeira é a quantificação do spread atualmente cedido a terceiros. Esta análise utiliza o framework de Total Cost of Ownership (TCO) para mapear o custo financeiro total da estrutura de recebimento. Ao decompor o spread por modalidade e volume transacional, identifica-se o custo financeiro real que a empresa-alvo está cedendo a terceiros e o impacto dessa estrutura sobre a rentabilidade líquida da operação."*

**Framework:** TCO — Custo Total de Propriedade invertido  
**Entrega:** Estimativa do spread por modalidade e volume, com série temporal

**Como este script responde à pergunta:**
> O script calcula o spread estimado capturado por cada modalidade — produto da receita, da taxa de desconto de mercado e do prazo de recebimento normalizado. Dois painéis respondem à pergunta:
> 1. **Spread total por modalidade (barras horizontais):** Ordenadas da menor para a maior contribuição em R$. A barra mais longa é a modalidade que mais spread cede a intermediários — e, portanto, onde há maior concentração de custo financeiro para a empresa-alvo.
> 2. **Evolução mensal do spread (barras + linha):** Barras de spread em R$ com linha sobreposta de percentual sobre a receita (eixo secundário). Se as barras crescem mas a linha de percentual oscila, o spread cresce em volume mas não em intensidade — sinal de que a estrutura de pagamento está estável. Se a linha sobe, os intermediários estão capturando uma fatia crescente do crescimento.

**Análise do Resultado:**
O volume total de spread cedido revela o tamanho da oportunidade de eficiência financeira disponível ao comprador. Se o spread cresce em volume absoluto mas se mantém estável como percentual da receita, a estrutura de pagamento está contida — os intermediários crescem junto com o negócio, mas não avançam sobre ele. Se a linha de percentual sobe ao longo do tempo, os intermediários estão capturando uma fatia crescente do crescimento, indicando perda progressiva de eficiência marginal. Esse dado define o teto de ganho operacional que a nova gestão pode perseguir ao renegociar condições com adquirentes.

In [ ]:
# ─── Spread por modalidade ────────────────────────────────────────────────────
receita_total = fin_fato["preco"].sum()
spread_total  = fin_fato["spread_intermediario"].sum()
meses_total   = fin_mensal["ano_mes"].nunique()

fig, axes = plt.subplots(1, 2, figsize=(16, 6))
fig.suptitle("Bloco 3 — Spread Capturado por Intermediários Financeiros",
             fontsize=13, fontweight="bold")

# Painel 1: Spread total por modalidade
fin_pag_ord = fin_pag.sort_values("spread_total", ascending=True)
bars = axes[0].barh(fin_pag_ord["tipo_pagamento"],
                    fin_pag_ord["spread_total"] / 1000,
                    color=COR_DESTAQUE, alpha=0.85)
axes[0].set_title("Spread Total por Modalidade (R$ mil)", fontsize=10)
axes[0].xaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f"R${x:,.0f}K"))
for bar, val in zip(bars, fin_pag_ord["spread_total"] / 1000):
    axes[0].text(bar.get_width() + 0.5, bar.get_y() + bar.get_height()/2,
                 f"R${val:,.0f}K", va="center", fontsize=9)

# Painel 2: Evolução mensal do spread
axes[1].bar(range(len(fin_mensal)), fin_mensal["spread_total"] / 1000,
            color=COR_DESTAQUE, alpha=0.8)
axes[1].plot(range(len(fin_mensal)), fin_mensal["pct_spread_receita"],
             color=COR_RECEITA, linewidth=2, marker="o", markersize=4,
             label="% Spread/Receita")
axes[1].set_xticks(range(len(fin_mensal)))
axes[1].set_xticklabels(fin_mensal["ano_mes"], rotation=45, fontsize=7)
axes[1].set_title("Evolução Mensal do Spread (R$ mil × % Receita)", fontsize=10)
axes[1].set_ylabel("Spread (R$ mil)")
axes[1].yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f"R${x:,.0f}K"))
axes[1].legend(fontsize=8)

plt.tight_layout()
salvar(fig, "03_spread_intermediarios")
plt.show()

print("\n── Spread por Modalidade ──────────────────────")
print(fin_pag[["tipo_pagamento", "receita_total", "spread_total", "pct_spread"]].to_string(index=False))
print(f"\n  Spread total do período : R$ {spread_total:,.0f}")
print(f"  Spread médio mensal     : R$ {spread_total/meses_total:,.0f}/mês")
print(f"  Spread como % da receita: {spread_total/receita_total*100:.2f}%")


---

## Análise 2 — Spread Líquido: Após Descontar o Custo do Risco

> *"A rentabilidade real da operação financeira só é visível após a dedução dos custos inerentes ao risco de crédito. Esta modelagem desconta sequencialmente do spread bruto a taxa de inadimplência (derivada do Bloco 2), o custo de oportunidade do capital imobilizado e as despesas operacionais fixas de gestão. O resultado é o spread líquido — o que a operação financeira efetivamente gera após cobrir todos os seus custos reais. Esse número responde se a estrutura atual é financeiramente sustentável ou se está operando com margem negativa mascarada pelo crescimento de receita."*

**Framework:** Break-even Analysis + TCO  
**Entrega:** Spread líquido por modalidade após desconto de risco e custo operacional

**Como este script responde à pergunta:**
> O script monta o cálculo completo do spread líquido mensal e compara modalidades. Dois painéis respondem à pergunta:
> 1. **Waterfall do spread mensal:** Partindo do spread bruto, subtrai sequencialmente o custo do risco (vermelho), o custo de capital (roxo) e o custo operacional fixo (cinza), chegando ao spread líquido — verde se positivo, vermelho se negativo. Cada barra é anotada com o valor em R$. Se a barra final for vermelha, o spread é insuficiente para cobrir os custos do risco nas premissas atuais.
> 2. **Spread bruto vs. líquido por modalidade:** Barras duplas lado a lado — laranja para bruto, verde ou vermelho para líquido — por modalidade de pagamento. Modalidades onde o líquido é negativo têm estrutura de custo de risco que supera o spread capturado — sinal de fragilidade financeira relevante para o comprador.

**Análise do Resultado:**
A transição do spread bruto para o líquido através do gráfico de cascata expõe a sensibilidade do lucro aos componentes de custo. Caso o spread líquido resulte negativo em certas modalidades, a manutenção do modelo terceirizado é a decisão técnica mais segura. Para o comprador, um spread líquido positivo indica que a estrutura financeira atual é sustentável e pode ser mantida ou otimizada pós-aquisição. Um spread líquido negativo sinaliza que o modelo de recebimento atual está destruindo valor — e que qualquer plano de crescimento precisa endereçar essa estrutura antes de comprometer capital adicional.


In [ ]:
# ─── Spread líquido por modalidade ───────────────────────────────────────────
capital_medio_aberto   = fin_fato["capital_em_aberto"].sum() / meses_total
custo_capital_mensal   = capital_medio_aberto * CUSTO_CAPITAL_AM
custo_risco_mensal     = receita_total / meses_total * TAXA_INADIMPLENCIA
spread_bruto_mensal    = spread_total / meses_total

# Recalcula para evitar conflito de nomes
custo_total_mes = custo_capital_mensal + CUSTO_OPERACIONAL_MENSAL + custo_risco_mensal
spread_liquido_mensal  = spread_bruto_mensal - custo_total_mes

# Por modalidade
df_modal_liq = fin_pag.copy()
df_modal_liq["custo_risco"] = df_modal_liq["receita_total"] * TAXA_INADIMPLENCIA
df_modal_liq["custo_capital"] = df_modal_liq["capital_aberto"] * CUSTO_CAPITAL_AM if "capital_aberto" in df_modal_liq.columns else df_modal_liq["spread_total"] * 0.2
df_modal_liq["custo_operacional"] = CUSTO_OPERACIONAL_MENSAL * (df_modal_liq["pct_receita"] / 100) * meses_total
df_modal_liq["spread_liquido"] = (df_modal_liq["spread_total"]
                                  - df_modal_liq["custo_risco"]
                                  - df_modal_liq["custo_operacional"])
df_modal_liq["pct_spread_liq"] = df_modal_liq["spread_liquido"] / df_modal_liq["receita_total"] * 100

fig, axes = plt.subplots(1, 2, figsize=(16, 6))
fig.suptitle("Bloco 3 — Spread Líquido: Bruto − Risco − Operacional",
             fontsize=13, fontweight="bold")

# Painel 1: Waterfall mensal — spread bruto → líquido
categorias_wf = ["Spread Bruto", "(-) Custo do Risco", "(-) Custo Capital", "(-) Custo Operacional", "= Spread Líquido"]
valores_wf    = [spread_bruto_mensal, -custo_risco_mensal, -custo_capital_mensal,
                 -CUSTO_OPERACIONAL_MENSAL, spread_liquido_mensal]
cores_wf      = [COR_DESTAQUE, COR_ALERTA, COR_ROXO, COR_NEUTRO,
                 COR_MARGEM if spread_liquido_mensal > 0 else COR_ALERTA]
bars = axes[0].bar(range(len(categorias_wf)), [abs(v)/1000 for v in valores_wf],
                   color=cores_wf, alpha=0.85)
axes[0].set_xticks(range(len(categorias_wf)))
axes[0].set_xticklabels(categorias_wf, rotation=20, fontsize=8)
axes[0].set_title("Decomposição do Spread Mensal (R$ mil)", fontsize=10)
axes[0].set_ylabel("R$ mil")
axes[0].yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f"R${x:,.0f}K"))
for bar, val in zip(bars, valores_wf):
    sinal = "" if val >= 0 else "−"
    axes[0].text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.1,
                 f"{sinal}R${abs(val)/1000:,.0f}K", ha="center", va="bottom",
                 fontsize=8, fontweight="bold")

# Painel 2: Spread bruto vs. líquido por modalidade
x = range(len(df_modal_liq))
w = 0.35
axes[1].bar([xi - w/2 for xi in x], df_modal_liq["spread_total"] / 1000,
            width=w, color=COR_DESTAQUE, alpha=0.7, label="Spread Bruto")
axes[1].bar([xi + w/2 for xi in x], df_modal_liq["spread_liquido"] / 1000,
            width=w,
            color=[COR_MARGEM if v > 0 else COR_ALERTA for v in df_modal_liq["spread_liquido"]],
            alpha=0.85, label="Spread Líquido")
axes[1].axhline(0, color="black", linewidth=0.5)
axes[1].set_xticks(list(x))
axes[1].set_xticklabels(df_modal_liq["tipo_pagamento"], rotation=20, fontsize=8)
axes[1].set_title("Spread Bruto vs. Líquido por Modalidade (R$ mil)", fontsize=10)
axes[1].yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f"R${x:,.0f}K"))
axes[1].legend(fontsize=8)

plt.tight_layout()
salvar(fig, "03_spread_liquido")
plt.show()

print(f"\n── Spread Líquido Mensal ──────────────────────")
print(f"  Spread bruto mensal       : R$ {spread_bruto_mensal:,.0f}")
print(f"  (-) Custo do risco         : R$ {custo_risco_mensal:,.0f}")
print(f"  (-) Custo de capital       : R$ {custo_capital_mensal:,.0f}")
print(f"  (-) Custo operacional      : R$ {CUSTO_OPERACIONAL_MENSAL:,.0f}")
print(f"  = Spread líquido mensal    : R$ {spread_liquido_mensal:,.0f}")
print(f"  Spread líquido / Receita   : {spread_liquido_mensal / (receita_total/meses_total) * 100:.2f}%")
print(f"  Viabilidade econômica      : {'✅ positivo' if spread_liquido_mensal > 0 else '🔴 negativo — internalização destrói valor'}")


---

## Análise 3 — Retorno Ajustado ao Risco e Custo de Oportunidade

> *"Decisões de investimento exigem a comparação do retorno projetado contra alternativas de mercado. Esta análise calcula o retorno anualizado da operação de crédito, ajustando-o pela volatilidade histórica do risco (Coeficiente de Variação). O retorno ajustado é então confrontado com benchmarks como CDI e Renda Fixa, enquanto a curva de payback estima o tempo necessário para recuperar o investimento inicial (CAPEX) em infraestrutura e sistemas de crédito."*

**Framework:** Análise de retorno ajustado ao risco + Custo de Oportunidade  
**Entrega:** Comparação de retorno ajustado ao risco da internalização versus alternativas

**Como este script responde à pergunta:**
> O script calcula o retorno anualizado da internalização, ajusta pela volatilidade do risco e confronta com benchmarks de mercado. Dois painéis respondem à pergunta:
> 1. **Retorno anualizado: internalização vs. alternativas:** Barras comparando o retorno ajustado ao risco da internalização contra CDI, renda fixa e FII médio — todos em percentual ao ano. A barra do retorno ajustado aparece em verde se supera o CDI (benchmark mínimo) ou vermelho se fica abaixo. A linha tracejada de 12% é o custo de oportunidade de referência.
> 2. **Curva de payback do investimento:** Linha que parte do investimento inicial negativo e sobe mês a mês pelo spread líquido. A linha tracejada verde marca o ponto de equilíbrio (saldo zero = payback). A linha vertical pontilhada vermelha indica o mês exato do payback, anotada na legenda. A área sombreada acima do zero destaca o período em que o investimento ainda não se pagou. Quanto mais íngreme a subida da curva, mais rapidamente o investimento se recupera.

**Análise do Resultado:**
O confronto com o custo de oportunidade define se o capital do comprador está sendo alocado de forma eficiente. Se o retorno ajustado não superar o CDI — benchmark mínimo de alocação em renda fixa no Brasil — a internalização não se justifica financeiramente mesmo que apresente lucro nominal, pois o risco assumido não é remunerado acima do custo de oportunidade. A curva de payback complementa essa visão ao definir a janela de maturação do projeto: o ponto exato onde o spread líquido acumulado cobre o investimento inicial. Quanto maior o spread líquido mensal e menor o CAPEX necessário, mais íngreme a curva — e mais rápido o comprador recupera o capital aportado na aquisição e nas melhorias operacionais planejadas.

In [ ]:
# ─── Retorno ajustado ao risco ────────────────────────────────────────────────
capital_necessario = fin_fato["capital_em_aberto"].sum() / meses_total + CUSTO_SETUP

# Retorno bruto da internalização (spread anualizado)
retorno_bruto_am   = spread_liquido_mensal / capital_necessario if capital_necessario > 0 else 0
retorno_bruto_aa   = (1 + retorno_bruto_am) ** 12 - 1

# Ajuste ao risco: desconto pela volatilidade (CV do risco do Bloco 2)
cv_risco           = fin_mensal["pct_anomalia"].std() / fin_mensal["pct_anomalia"].mean() \
                     if fin_mensal["pct_anomalia"].mean() > 0 else 0
retorno_ajustado   = retorno_bruto_aa * (1 - cv_risco)

# Alternativas de alocação (benchmarks mercado brasileiro 2025)
# Fontes: 
# 1. CDI: Relatório Focus/BCB (Expectativa SELIC 2025 média 11.75% - 12.25%)
# 2. Renda Fixa: Curva longa de juros (NTN-B 2025/2030 + IPCA)
# 3. FIIs: Dividend Yield médio do IFIX (Indice de Fundos de Investimento Imobiliários)

alternativas = {
    "CDI (12.25%)"        : 0.1225,  # Baseado na manutenção da Selic (Relatório Focus/BCB 2025)
    "Renda Fixa (13.5%)"  : 0.1350,  # Crédito Privado (Debêntures/CDBs) pós-fixados
    "FII Médio (10.5%)"   : 0.1050,  # Dividend Yield médio estimado para FIIs de tijolo/papel 2025
    "Internalizar Cred." : retorno_ajustado,
}

fig, axes = plt.subplots(1, 2, figsize=(14, 6))
fig.suptitle("Bloco 3 — Retorno Ajustado ao Risco vs. Custo de Oportunidade",
             fontsize=13, fontweight="bold")

# Painel 1: Comparação de retornos anualizados
cores_alt = [COR_NEUTRO, COR_NEUTRO, COR_NEUTRO,
             COR_MARGEM if retorno_ajustado > 0.12 else COR_ALERTA]
bars = axes[0].bar(list(alternativas.keys()),
                   [v * 100 for v in alternativas.values()],
                   color=cores_alt, alpha=0.85)
axes[0].axhline(12, color=COR_ALERTA, linestyle="--", linewidth=1.5,
                alpha=0.6, label="CDI benchmark")
axes[0].set_title("Retorno Anualizado: Internalização vs. Alternativas (%)", fontsize=10)
axes[0].yaxis.set_major_formatter(mticker.FuncFormatter(fmt_pct))
axes[0].legend(fontsize=8)
axes[0].tick_params(axis="x", rotation=15)
for bar, val in zip(bars, alternativas.values()):
    axes[0].text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.2,
                 f"{val*100:.1f}%", ha="center", va="bottom", fontsize=9, fontweight="bold")

# Painel 2: Payback do setup
payback_meses = CUSTO_SETUP / spread_liquido_mensal if spread_liquido_mensal > 0 else float("inf")
payback_linha = [CUSTO_SETUP - spread_liquido_mensal * m for m in range(min(37, int(payback_meses * 2) + 12))]
payback_linha = [v for v in payback_linha if v > -CUSTO_SETUP]
axes[1].plot(range(len(payback_linha)), [v / 1000 for v in payback_linha],
             color=COR_RECEITA, linewidth=2)
axes[1].axhline(0, color=COR_MARGEM, linestyle="--", linewidth=1.5, label="Payback")
if spread_liquido_mensal > 0:
    axes[1].axvline(payback_meses, color=COR_ALERTA, linestyle=":", linewidth=1.5,
                    label=f"Payback: {payback_meses:.1f} meses")
axes[1].fill_between(range(len(payback_linha)), [v / 1000 for v in payback_linha],
                     0, where=[v > 0 for v in payback_linha],
                     alpha=0.1, color=COR_ALERTA)
axes[1].set_title("Curva de Payback do Investimento em Crédito Próprio", fontsize=10)
axes[1].set_xlabel("Meses")
axes[1].set_ylabel("Saldo (R$ mil)")
axes[1].yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f"R${x:,.0f}K"))
axes[1].legend(fontsize=8)

plt.tight_layout()
salvar(fig, "03_retorno_ajustado")
plt.show()

print(f"\n── Retorno Ajustado ao Risco ──────────────────────")
print(f"  Capital necessário estimado  : R$ {capital_necessario:,.0f}")
print(f"  Retorno bruto anualizado     : {retorno_bruto_aa*100:.1f}%")
print(f"  CV do risco (ajuste)         : {cv_risco:.2f}")
print(f"  Retorno ajustado ao risco    : {retorno_ajustado*100:.1f}%")
print(f"  CDI benchmark                : 12,0%")
print(f"  Vantagem vs. CDI             : {(retorno_ajustado - 0.12)*100:+.1f}pp")
if spread_liquido_mensal > 0:
    print(f"  Payback do setup             : {payback_meses:.1f} meses")
else:
    print(f"  Payback do setup             : indefinido — spread líquido negativo")


---

## Análise 4 — Mapa de Deterioração da Rentabilidade

> *"Modelos financeiros são sensíveis a variações extremas de mercado. Através de uma Análise de Sensibilidade, este mapa de calor (Heatmap) estressa simultaneamente as variáveis de volume de vendas e taxa de inadimplência. O objetivo é identificar os "pontos de ruptura" — cenários específicos onde a margem do spread líquido entra em terreno negativo, servindo como um guia de gestão de riscos para o monitoramento pós-aquisição."*

**Framework:** Análise de Sensibilidade + Planejamento por Cenários  
**Entrega:** Mapa de calor de rentabilidade variando inadimplência e volume

**Como este script responde à pergunta:**
> O script constrói uma matriz 6×6 variando inadimplência e volume simultaneamente e calcula a margem do spread líquido para cada combinação. Um único painel responde à pergunta:
> 1. **Heatmap inadimplência × volume:** Cada célula mostra a margem percentual do spread líquido — verde para positivo (criação de valor), vermelho para negativo (destruição de valor). A linha da taxa de inadimplência atual e a coluna do volume atual marcam o ponto de partida real. Os quadrantes vermelhos à esquerda (volume baixo) ou acima (inadimplência alta) são os gatilhos de risco — as condições em que o spread se torna negativo e a estrutura financeira passa a destruir valor.

**Análise do Resultado:**
O heatmap traduz a análise de sensibilidade em linguagem de gestão de risco: cada célula vermelha é um cenário que o comprador precisa monitorar ativamente no pós-aquisição. O ponto de partida real — cruzamento entre a taxa de inadimplência atual e o volume atual — posiciona o ativo no mapa e revela sua distância dos quadrantes de destruição de valor. Se o ponto atual está próximo da fronteira entre verde e vermelho, a margem de segurança é baixa e qualquer deterioração moderada de crédito ou queda de volume cruza o limiar de ruptura. Se está no centro do quadrante verde, o ativo tem resiliência para absorver choques sem comprometer a rentabilidade projetada. Este gráfico deve ser o painel de monitoramento dos primeiros 12 meses pós-fechamento.

In [ ]:
# ─── Mapa de calor: inadimplência × volume ────────────────────────────────────
receita_base_mensal = receita_total / meses_total
spread_base_pct     = spread_total / receita_total  # spread como % da receita

fatores_volume = [0.70, 0.85, 1.00, 1.15, 1.30, 1.50]
taxas_inad     = [TAXA_INADIMPLENCIA * m for m in [0.5, 0.75, 1.0, 1.5, 2.0, 3.0]]

matriz_margem = []
for taxa in taxas_inad:
    linha = []
    for fator_vol in fatores_volume:
        receita_m      = receita_base_mensal * fator_vol
        spread_m       = receita_m * spread_base_pct
        custo_risco_m  = receita_m * taxa
        custo_cap_m    = (fin_fato["capital_em_aberto"].sum() / meses_total * fator_vol) * CUSTO_CAPITAL_AM
        spread_liq_m   = spread_m - custo_risco_m - custo_cap_m - CUSTO_OPERACIONAL_MENSAL
        margem_pct     = spread_liq_m / receita_m * 100
        linha.append(round(margem_pct, 2))
    matriz_margem.append(linha)

df_heatmap = pd.DataFrame(
    matriz_margem,
    index=[f"{t*100:.2f}%" for t in taxas_inad],
    columns=[f"×{f:.2f}" for f in fatores_volume]
)

fig, ax = plt.subplots(figsize=(12, 7))
sns.heatmap(df_heatmap, annot=True, fmt=".1f", cmap="RdYlGn",
            linewidths=0.5, ax=ax, center=0,
            cbar_kws={"label": "Margem do Spread Líquido (%)"})
ax.set_title("Bloco 3 — Mapa de Deterioração da Rentabilidade\n"
             "(Inadimplência × Volume | verde = positivo, vermelho = destruidor)",
             fontsize=12, fontweight="bold")
ax.set_xlabel("Fator de Volume (×base)")
ax.set_ylabel("Taxa de Inadimplência")
plt.tight_layout()
salvar(fig, "03_mapa_deterioracao")
plt.show()

print("\n── Leitura do Mapa ──────────────────────")
print(f"  Taxa de inadimplência base: {TAXA_INADIMPLENCIA*100:.2f}%")
print(f"  Volume base mensal        : R$ {receita_base_mensal:,.0f}")
print(f"  Células vermelhas (marg<0): quadrante de destruição de valor")
print(f"  Células verdes (marg>0)   : quadrante de criação de valor")
print(f"  Gatilho de saída          : inadimplência acima de {TAXA_INADIMPLENCIA*200:.2f}% ou "
      f"volume abaixo de 70% do base")


In [ ]:
# ─── Score do Bloco 3 ────────────────────────────────────────────────────────
_b3_spread_positivo    = spread_liquido_mensal > 0
_b3_retorno_adequado   = retorno_ajustado > 0.10
_b3_payback_aceitavel  = payback_meses <= 24 if spread_liquido_mensal > 0 else False

if not _b3_spread_positivo:
    _b3_score = 1
    _b3_sinal = "🔴 Spread líquido negativo — estrutura financeira destrói valor com as premissas atuais"
    _b3_cor   = COR_ALERTA
elif _b3_spread_positivo and _b3_retorno_adequado and _b3_payback_aceitavel:
    _b3_score = 3
    _b3_sinal = "✅ Spread líquido positivo e retorno acima do custo de capital — perfil financeiro favorável"
    _b3_cor   = COR_MARGEM
else:
    _b3_score = 2
    _b3_sinal = "⚠️  Spread positivo mas retorno marginal — requer avaliação com premissas conservadoras"
    _b3_cor   = COR_DESTAQUE

_b3_cond = (
    f"Revisitar premissas de custo operacional (R$ {CUSTO_OPERACIONAL_MENSAL:,.0f}/mês) "
    f"e custo de capital ({CUSTO_CAPITAL_AM*100:.1f}%a.m.) antes de qualquer compromisso."
    if _b3_score <= 2
    else "Rentabilidade sustenta a tese de aquisição sob as premissas atuais."
)

print("=" * 60)
print("SÍNTESE — BLOCO 3: RENTABILIDADE FINANCEIRA")
print("=" * 60)
print(f"  Spread bruto total (período): R$ {spread_total:,.0f}")
print(f"  Spread bruto médio mensal   : R$ {spread_bruto_mensal:,.0f}")
print(f"  Spread líquido mensal       : R$ {spread_liquido_mensal:,.0f}")
print(f"  Retorno ajustado ao risco   : {retorno_ajustado*100:.1f}% a.a.")
if spread_liquido_mensal > 0:
    print(f"  Payback do investimento     : {payback_meses:.1f} meses")
else:
    print(f"  Payback                     : indefinido (spread negativo)")
print()
print(f"  Score do Bloco: {_b3_score}/3")
print(f"  Sinal        : {_b3_sinal}")
print()
print(f"  Condicionante: {_b3_cond}")
print("=" * 60)


---
*Próximo notebook: `04_capital_escalabilidade.ipynb` — A estrutura financeira da empresa-alvo é compatível com crescimento — e o modelo de recebimento escala junto com o volume operacional?*

> Esta análise faz parte do **Projeto Fictus**, conduzido pela Lufi Data Consulting. Os três módulos analíticos — Vendas, Logística e Finanças — compõem a base do Relatório de Recomendação de Aquisição.
